# **Data Cleaning**

In [14]:
# Load pandas for data processing and datetime utilities for any date operations.
import pandas as pd

# Set option to display all columns and format float values to two decimal places (to avoid scientific notation).
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [15]:
# Load the raw e-commerce dataset for cleaning.
ecommerce_df = pd.read_csv(r'dataset/raw/ecommerce_dataset_+1m.csv')

## Initial Cleaning

Goals for this section:
- Remove unnecessary / redundant columns
- Inspect data quality (nulls, sample, dtypes)
- Round float columns to 2 decimal places
- Flag logically invalid values

In [16]:
# Display all the columns
ecommerce_df.columns

Index(['order_id', 'order_date', 'order_year', 'order_month', 'order_day',
       'order_hour', 'order_minute', 'order_second', 'is_weekend',
       'order_status', 'return_reason', 'customer_id', 'customer_name',
       'gender', 'age', 'customer_segment', 'country', 'city',
       'customer_loyalty_score', 'total_orders_by_customer',
       'account_creation_date', 'product_id', 'product_name', 'category',
       'sub_category', 'brand', 'product_rating_avg', 'product_reviews_count',
       'stock_quantity', 'unit_price_usd', 'quantity', 'discount_percent',
       'discount_amount_usd', 'total_price_usd', 'cost_usd', 'profit_usd',
       'tax_usd', 'currency', 'payment_method', 'payment_status',
       'installment_plan', 'shipping_method', 'shipping_cost_usd',
       'delivery_days', 'shipping_country', 'warehouse_location',
       'delivery_status', 'rating', 'review_sentiment', 'customer_feedback',
       'coupon_used', 'coupon_code', 'campaign_source', 'device_type',
       'traf

### 1. Remove unnecessary columns

In [17]:
# Eliminate columns that are not relevant to the analysis or contain redundant information.

ecommerce_df = ecommerce_df[
    [
        # TIME & CONTEXT    
        'order_date',
        'order_year',
        'order_month',
        'is_weekend',
        
        # CUSTOMER DEMOGRAPHICS / GEOGRAPHY
        'customer_name',
        'gender',
        'age',
        'customer_segment',
        'country',
        
        # PRODUCT
        'order_status',
        'category',
        'sub_category',
        'unit_price_usd',
        'quantity',
        
        # FINANCIALS
        'discount_percent',
        'total_price_usd',
        'profit_usd',
        'profit_margin_percent',
        
        # PAYMENT & SHIPPING
        'payment_method',
        'shipping_method',
        'shipping_cost_usd',
        'delivery_days',
        'shipping_country',
        
        # CUSTOMER BEHAVIOR
        'rating',
        'customer_loyalty_score',
        'coupon_used',
        'session_duration_minutes',
        'pages_visited',
        'abandoned_cart_before',
        
        # RISK & PERFORMANCE
        'fraud_risk_score',
        'device_type',
        
        # MARKETING
        'campaign_source',
        'traffic_source',
    ]
].copy()

### 2. Rename Columns

In [18]:
ecommerce_df = ecommerce_df.rename(
    columns={
        'total_price_usd' : 'revenue_usd',
        'session_duration_minutes' : 'session_duration_min'
    }
)

### 3. Standardize Columns

In [19]:
ecommerce_df['order_date'] = pd.to_datetime(ecommerce_df['order_date'], errors='coerce')

### 4. Data quality check

In [20]:
ecommerce_df.info()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000123 entries, 0 to 1000122
Data columns (total 33 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   order_date              1000123 non-null  datetime64[ns]
 1   order_year              1000123 non-null  int64         
 2   order_month             1000123 non-null  int64         
 3   is_weekend              1000123 non-null  object        
 4   customer_name           1000123 non-null  object        
 5   gender                  1000123 non-null  object        
 6   age                     1000123 non-null  int64         
 7   customer_segment        1000123 non-null  object        
 8   country                 1000123 non-null  object        
 9   order_status            1000123 non-null  object        
 10  category                1000123 non-null  object        
 11  sub_category            1000123 non-null  object        
 12  unit_price_usd

In [21]:
# Missing values (%)
print(((ecommerce_df.isnull().sum() / len(ecommerce_df)) * 100).round(2).to_string())

order_date               0.00
order_year               0.00
order_month              0.00
is_weekend               0.00
customer_name            0.00
gender                   0.00
age                      0.00
customer_segment         0.00
country                  0.00
order_status             0.00
category                 0.00
sub_category             0.00
unit_price_usd           0.00
quantity                 0.00
discount_percent         0.00
revenue_usd              0.00
profit_usd               0.00
profit_margin_percent    0.00
payment_method           0.00
shipping_method          0.00
shipping_cost_usd        0.00
delivery_days            0.00
shipping_country         0.00
rating                   0.00
customer_loyalty_score   0.00
coupon_used              0.00
session_duration_min     0.00
pages_visited            0.00
abandoned_cart_before    0.00
fraud_risk_score         0.00
device_type              0.00
campaign_source          0.00
traffic_source           0.00


In [22]:
# Random sample
ecommerce_df.sample(10)

,order_date,order_year,order_month,is_weekend,customer_name,gender,age,customer_segment,country,order_status,category,sub_category,unit_price_usd,quantity,discount_percent,revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,delivery_days,shipping_country,rating,customer_loyalty_score,coupon_used,session_duration_min,pages_visited,abandoned_cart_before,fraud_risk_score,device_type,campaign_source,traffic_source
293437,2024-06-30 13:52:20.274188,2024,6,Yes,Dawn Nelson,Female,52,Premium,Belgium,Completed,Sports,Gym Equipment,139.65,3,20,335.16,115.08,34.34,Bank Transfer,Economy,1.35,1,Belgium,1,34.90,No,18.80,19,Yes,25.10,Mobile,Google Ads,Social
116450,2025-08-02 05:18:36.175105,2025,8,Yes,Valerie Gallagher,Female,34,Regular,Spain,Completed,Electronics,Cameras,335.37,4,25,"1,006.11",304.35,30.25,PayPal,Next Day,6.33,12,Spain,4,76.50,No,55.40,19,Yes,51.40,Mobile,Affiliate,Email
202505,2024-03-19 20:28:29.424365,2024,3,No,Aaron Jenkins,Male,50,Regular,Germany,Pending,Home,Furniture,142.74,4,20,456.77,158.45,34.69,Apple Pay,Next Day,4.62,3,Germany,4,13.50,No,30.00,19,Yes,16.00,Tablet,Affiliate,Search
7806,2024-07-17 12:03:55.286567,2024,7,No,Jonathan Nguyen,Male,66,VIP,Spain,Completed,Sports,Accessories,114.31,4,10,411.52,174.36,42.37,Bank Transfer,Next Day,17.11,3,Spain,4,80.40,No,46.30,1,Yes,5.80,Desktop,Facebook,Direct
762771,2024-06-10 18:57:33.922690,2024,6,No,David Campbell,Male,62,Premium,Italy,Completed,Home,Appliances,298.30,4,20,954.56,349.60,36.62,PayPal,Economy,16.47,5,Italy,3,97.20,Yes,21.10,10,No,26.90,Tablet,Instagram,Email
539403,2024-09-22 02:44:55.541265,2024,9,Yes,Bailey Hall,Female,20,Regular,United Kingdom,Cancelled,Health,Medical Devices,47.87,5,5,227.38,69.48,30.56,Bank Transfer,Standard,15.07,6,United Kingdom,1,72.50,Yes,49.80,16,Yes,55.60,Mobile,Instagram,Search
248134,2025-06-19 09:33:22.427322,2025,6,No,David Murray,Male,30,Regular,Netherlands,Completed,Clothing,Shoes,80.39,1,10,72.35,38.46,53.16,PayPal,Express,11.82,10,Netherlands,3,10.90,No,18.40,4,Yes,47.00,Desktop,Instagram,Direct
384871,2025-01-30 10:47:56.208368,2025,1,No,Sean Johns,Male,61,Regular,Germany,Completed,Home,Furniture,82.65,3,20,198.36,82.86,41.77,Debit Card,Economy,18.45,6,Germany,1,93.60,No,59.30,12,Yes,64.20,Desktop,Instagram,Direct
744365,2024-08-10 17:12:13.936736,2024,8,Yes,Robert Edwards,Male,71,VIP,United States,Completed,Sports,Accessories,245.87,1,10,221.28,99.75,45.08,Apple Pay,Standard,9.09,6,United States,3,22.00,Yes,51.80,3,No,83.90,Mobile,Facebook,Search
239598,2025-05-23 15:05:13.293481,2025,5,No,Mark Sanchez,Male,51,Regular,Belgium,Pending,Clothing,Kids Wear,137.09,5,10,616.90,298.45,48.38,Debit Card,Express,19.00,13,Belgium,4,32.90,Yes,19.60,13,No,61.50,Mobile,Organic,Referral


### 5. Round float columns to 2 decimal places

In [23]:
# Round float values to two decimal places for cleaner output.
float_cols = ecommerce_df.select_dtypes('float')

for col in float_cols.columns:
    ecommerce_df[col] = ecommerce_df[col].round(2)

### 6. Flag logically invalid values

In [24]:
# 1. Age
invalid_age = ecommerce_df[(ecommerce_df['age'] < 0) | (ecommerce_df['age'] > 120)]

# 2. Quantity
invalid_quantity = ecommerce_df[ecommerce_df['quantity'] <= 0]

# 3. Discount
invalid_discount = ecommerce_df[
    (ecommerce_df['discount_percent'] < 0) | (ecommerce_df['discount_percent'] > 100)
]

# 4. Rating (fix this!)
invalid_rating = ecommerce_df[(ecommerce_df['rating'] < 1) | (ecommerce_df['rating'] > 5)]

# 5. Delivery days
invalid_delivery = ecommerce_df[
    (ecommerce_df['delivery_days'] < 0) | (ecommerce_df['delivery_days'] > 60)
]

# 6. Monetary values
invalid_money = ecommerce_df[
    (ecommerce_df['unit_price_usd'] < 0) |
    (ecommerce_df['revenue_usd'] < 0) |
    (ecommerce_df['shipping_cost_usd'] < 0)
]

# 7. Fraud score
invalid_fraud = ecommerce_df[
    (ecommerce_df['fraud_risk_score'] < 0) | (ecommerce_df['fraud_risk_score'] > 100)
]

# Print summary
print(f'Invalid Age        : {len(invalid_age)}')
print(f'Invalid Quantity   : {len(invalid_quantity)}')
print(f'Invalid Discount   : {len(invalid_discount)}')
print(f'Invalid Rating     : {len(invalid_rating)}')
print(f'Invalid Delivery   : {len(invalid_delivery)}')
print(f'Invalid Money      : {len(invalid_money)}')
print(f'Invalid Fraud      : {len(invalid_fraud)}')

Invalid Age        : 0
Invalid Quantity   : 0
Invalid Discount   : 0
Invalid Rating     : 0
Invalid Delivery   : 0
Invalid Money      : 0
Invalid Fraud      : 0


### 7. Save cleaned dataset

In [25]:
# Save the cleaned dataset to a new CSV file.
ecommerce_df.to_csv("dataset/cleaned/ecommerce_cleaned.csv", index=False)